# ScanOps 보안 모델 — 클라우드 GPU 학습 + 평가 (올인원)

Mac에서 ~3시간 걸리는 QLoRA 학습을 Colab 무료 **T4 GPU**에서 진짜 4bit QLoRA로 **약 5~10분**에 끝내고, **OWASP 외부 벤치마크 평가(F1/오탐률/CWE정확도)까지 클라우드에서 바로** 확인합니다.

**준비:** 런타임 → 런타임 유형 변경 → **T4 GPU** 선택.

**업로드할 파일 2개** (로컬 `scanops-model/data/`):
- `lora_train_v8.jsonl` — 학습 데이터
- `owasp_holdout_eval.json` — OWASP 외부 평가 홀드아웃 110케이스

흐름: ① 설치 → ② 업로드 → ③ 학습 → ④ 평가 → ⑤ 학습곡선·어댑터 다운로드

## ① 의존성 설치 + GPU 확인

In [ ]:
!pip -q install transformers peft datasets accelerate bitsandbytes matplotlib
import torch
assert torch.cuda.is_available(), '런타임 유형을 T4 GPU로 바꾸세요.'
print('GPU:', torch.cuda.get_device_name(0))

## ② 파일 2개 업로드 (lora_train_v8.jsonl, owasp_holdout_eval.json)

In [ ]:
from google.colab import files
up = files.upload()
assert 'lora_train_v8.jsonl' in up and 'owasp_holdout_eval.json' in up, '두 파일 모두 올려주세요.'
print('업로드 완료:', list(up.keys()))

## ③ QLoRA 학습 (진짜 4bit, CUDA)
로컬 `ml/train.py`와 동일한 방법: Qwen2.5-Coder-1.5B + LoRA(r=32) + cross-entropy + AdamW/cosine. 하이퍼파라미터만 바꿔 실험하세요(EPOCHS, R, LR).

In [ ]:
import json, time, torch
from datasets import Dataset
from peft import LoraConfig, TaskType, get_peft_model, prepare_model_for_kbit_training
from transformers import (AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig,
                          DataCollatorForSeq2Seq, Trainer, TrainingArguments)

BASE = 'Qwen/Qwen2.5-Coder-1.5B-Instruct'
MAXLEN, R, ALPHA, EPOCHS, LR = 1024, 32, 64, 4, 1e-4   # ← 실험: EPOCHS/R/LR 조정

rows = [json.loads(l) for l in open('lora_train_v8.jsonl') if l.strip()]
n_none = sum(1 for r in rows if r['completion'].split(chr(10))[0].upper().startswith('VULNERABILITY: NONE'))
print(f'{len(rows)}개 예시, 안전(NONE) {100*n_none/len(rows):.1f}%')
ds = Dataset.from_list(rows).train_test_split(test_size=0.1, seed=42)

tok = AutoTokenizer.from_pretrained(BASE, trust_remote_code=True)
if tok.pad_token is None: tok.pad_token = tok.eos_token
SYS = 'You are a security code analyzer.'
def fmt(e):
    return {'text': f'<|im_start|>system\n{SYS}<|im_end|>\n<|im_start|>user\n'+e['prompt']+
            '<|im_end|>\n<|im_start|>assistant\n'+e['completion']+'<|im_end|>'}
def tk(e):
    o = tok(e['text'], truncation=True, max_length=MAXLEN); o['labels']=o['input_ids'].copy(); return o
tr = ds['train'].map(fmt).map(tk, remove_columns=ds['train'].column_names+['text'])
ev = ds['test'].map(fmt).map(tk, remove_columns=ds['test'].column_names+['text'])

bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type='nf4',
                         bnb_4bit_compute_dtype=torch.bfloat16, bnb_4bit_use_double_quant=True)
model = AutoModelForCausalLM.from_pretrained(BASE, quantization_config=bnb, device_map='auto', trust_remote_code=True)
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, LoraConfig(r=R, lora_alpha=ALPHA, lora_dropout=0.05,
        target_modules=['q_proj','k_proj','v_proj','o_proj'], task_type=TaskType.CAUSAL_LM))
model.print_trainable_parameters()

args = TrainingArguments(output_dir='out', num_train_epochs=EPOCHS, per_device_train_batch_size=4,
        gradient_accumulation_steps=2, learning_rate=LR, lr_scheduler_type='cosine', warmup_ratio=0.03,
        fp16=True, logging_steps=10, eval_strategy='epoch', save_strategy='epoch',
        load_best_model_at_end=True, report_to='none')
trainer = Trainer(model=model, args=args, train_dataset=tr, eval_dataset=ev,
        data_collator=DataCollatorForSeq2Seq(tok, pad_to_multiple_of=8, return_tensors='pt'))
t0=time.time(); trainer.train(); print(f'학습 {(time.time()-t0)/60:.1f}분')
model.save_pretrained('adapter'); tok.save_pretrained('adapter')
LOSSLOG=[{'step':e['step'],'loss':e.get('loss'),'eval_loss':e.get('eval_loss')} for e in trainer.state.log_history if 'loss' in e or 'eval_loss' in e]
json.dump(LOSSLOG, open('adapter/train_loss.json','w'), indent=2)

## ④ OWASP 외부 벤치마크 평가 (클라우드에서 바로)
방금 학습한 어댑터로 OWASP 홀드아웃 110케이스를 추론하고 F1/정밀도/재현율/오탐률/CWE정확도를 계산합니다. (병합 모델로 GPU 추론 → 빠름)

In [ ]:
import re, json
merged = model.merge_and_unload()   # LoRA 병합 후 추론
merged.eval()
cases = json.load(open('owasp_holdout_eval.json'))

OUTPUT_FORMAT = ('First decide whether the code has a REAL, exploitable security vulnerability.\n'
  'If the code is SAFE (parameterized queries, output escaping, input validation, strong crypto/hash, '
  'secure randomness, proper auth), respond with EXACTLY one line:\nVULNERABILITY: NONE\n'
  'If it DOES have a vulnerability, list ALL. For EACH use this format separated by ---:\n'
  'VULNERABILITY: [name with CWE ID]\nSEVERITY: [CRITICAL/HIGH/MEDIUM/LOW]\nCVSS: [score]\n'
  'ATTACK: [한 문장]\nFIX: [수정]\n---')
HINT={'Java':'java'}
def prompt(c):
    return (f"Analyze this {c['language']} code for security vulnerabilities:\n\n"
            f"```{HINT.get(c['language'],'text')}\n{c['code']}\n```\n\n{OUTPUT_FORMAT}")
def gen(c):
    msgs=[{'role':'system','content':SYS},{'role':'user','content':prompt(c)}]
    ids=tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors='pt').to(merged.device)
    out=merged.generate(ids, max_new_tokens=120, do_sample=False, pad_token_id=tok.eos_token_id)
    return tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True)
def vuln_of(txt):
    m=re.search(r'VULNERABILITY:\s*(.+)', txt); return m.group(1).strip() if m else ''
def is_safe(v): return (not v) or v.upper().startswith('NONE')
CAT_KW={'sqli':['sql','89'],'xss':['xss','cross-site','scripting','79','80'],'cmdi':['command','78','77'],
 'pathtraver':['traversal','22','23','36'],'crypto':['crypto','encrypt','327','cipher',' des','broken'],
 'hash':['hash','md5','sha-1','sha1','328','326'],'ldapi':['ldap','90'],'xpathi':['xpath','643'],
 'trustbound':['trust bound','501'],'securecookie':['cookie','secure flag','614','311','315','1004'],
 'weakrand':['random','330','338','predictable']}
def cwe_ok(v,cat): t=v.lower(); return any(k in t for k in CAT_KW.get(cat,[]))

tp=fn=fp=tn=cwe=0
for i,c in enumerate(cases):
    v=vuln_of(gen(c)); flagged=not is_safe(v); isv=c['label']=='vuln'
    if isv and flagged: tp+=1; cwe+= cwe_ok(v,c['category'])
    elif isv and not flagged: fn+=1
    elif not isv and flagged: fp+=1
    else: tn+=1
    if i%20==0: print(f'  {i}/{len(cases)}')
rec=tp/(tp+fn)*100; fpr=fp/(fp+tn)*100; prec=tp/(tp+fp)*100 if tp+fp else 0
acc=(tp+tn)/len(cases)*100; f1=2*prec*rec/(prec+rec) if prec+rec else 0
print('='*60)
print(f'ScanOps(클라우드): F1={f1:.1f} 정밀도={prec:.1f}% 재현율={rec:.1f}% 오탐률={fpr:.1f}% 정확도={acc:.1f}% CWE정확={cwe/(tp+fn)*100:.1f}%')
print(f'  TP={tp} FN={fn} FP={fp} TN={tn}')
print('비교) Grok-3-mini: F1=62.9 정밀도=66.0% 재현율=60.0% 오탐률=30.9% 정확도=64.5% CWE정확=60.0%')
print('='*60)

## ⑤ 학습곡선 + 어댑터 다운로드
- 그래프로 수렴/과적합 확인
- 결과가 좋으면 `adapter.zip`을 로컬 `models/qwen-security-qlora-v8/`에 풀고 `scripts/convert_to_gguf_v5.py`(태그 v8)로 GGUF 변환·서빙
- 결과가 아쉬우면 ③에서 EPOCHS/R/LR 바꿔 다시 (10분이면 끝)

In [ ]:
import matplotlib.pyplot as plt
tr=[(e['step'],e['loss']) for e in LOSSLOG if e.get('loss') is not None]
evl=[(e['step'],e['eval_loss']) for e in LOSSLOG if e.get('eval_loss') is not None]
plt.figure(figsize=(7,4))
if tr: plt.plot(*zip(*tr),'-o',ms=3,label='train loss')
if evl: plt.plot(*zip(*evl),'-s',label='eval loss')
plt.xlabel('step'); plt.ylabel('cross-entropy loss'); plt.legend(); plt.grid(alpha=.4)
plt.title('학습곡선 (QLoRA v8)'); plt.savefig('learning_curve.png',dpi=150,bbox_inches='tight'); plt.show()

!cd adapter && zip -r ../adapter_v8.zip . >/dev/null
from google.colab import files
files.download('adapter_v8.zip')
files.download('learning_curve.png')